# 06 - Entrainement complet des modeles

Dans ce notebook, on refait tout l'entrainement : chargement des donnees, preparation, split train/test, creation des pipelines, evaluation et sauvegarde des modeles.

L'objectif est que le travail soit lisible directement dans le notebook, sans passer par un script cache.

## 1. Imports et chemins

In [ ]:
from pathlib import Path
from time import perf_counter

import joblib
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import sys
sys.path.append(str(BASE_DIR))
from feature_engineering import prepare_customer_features

DATA_PATH = BASE_DIR / 'data' / 'customer_churn.csv'
MODELS_DIR = BASE_DIR / 'models'
REPORTS_DIR = BASE_DIR / 'reports'
PREPROCESSED_PATH = BASE_DIR / 'data_preprocessed.pkl'

RANDOM_STATE = 42
THRESHOLD = 0.35
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

## 2. Chargement et preparation metier

On applique les variables metier creees dans `feature_engineering.py` : satisfaction, engagement, risque paiement, pression support, etc.

In [ ]:
raw_df = pd.read_csv(DATA_PATH)
df = prepare_customer_features(raw_df)

X = df.drop(columns=['churn', 'customer_id'])
y = df['churn']

print('Shape X :', X.shape)
print('Taux de churn :', round(y.mean(), 3))

## 3. Split stratifie

On utilise un split stratifie pour garder la meme proportion de churners dans le train et dans le test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print('Train :', X_train.shape)
print('Test :', X_test.shape)
print('Churn train :', round(y_train.mean(), 3))
print('Churn test :', round(y_test.mean(), 3))

## 4. Sauvegarde des infos utiles

Le dashboard aura besoin des colonnes, des medianes et du jeu de test pour calculer les indicateurs.

In [ ]:
data_info = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'all_cols': X.columns.tolist(),
    'input_cols': raw_df.drop(columns=['churn', 'customer_id']).columns.tolist(),
    'medians': X.median(numeric_only=True).to_dict(),
    'modes': X.mode(dropna=True).iloc[0].to_dict(),
    'threshold_recommande': THRESHOLD,
}
joblib.dump(data_info, PREPROCESSED_PATH)
print('Fichier sauvegarde :', PREPROCESSED_PATH)

## 5. Preprocessing adapte aux modeles

La regression logistique et le MLP utilisent un `StandardScaler`, car ils sont sensibles aux echelles.

Random Forest et XGBoost gardent les variables numeriques telles quelles, car les arbres n'ont pas besoin de standardisation.

In [ ]:
def make_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

pre_scaled = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', make_encoder(), cat_cols),
])

pre_unscaled = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', make_encoder(), cat_cols),
])

## 6. Creation des 4 modeles

On compare 4 familles : baseline lineaire, arbres, boosting, et Deep Learning.

In [ ]:
models = {
    'LogisticRegression': Pipeline([
        ('pre', pre_scaled),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    'RandomForest': Pipeline([
        ('pre', pre_unscaled),
        ('model', RandomForestClassifier(
            n_estimators=250,
            max_depth=10,
            min_samples_leaf=2,
            class_weight='balanced_subsample',
            n_jobs=1,
            random_state=RANDOM_STATE,
        )),
    ]),
    'XGBoost': Pipeline([
        ('pre', pre_unscaled),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            scale_pos_weight=12,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
        )),
    ]),
    'DeepLearning': Pipeline([
        ('pre', pre_scaled),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', MLPClassifier(
            hidden_layer_sizes=(64, 32),
            alpha=0.001,
            early_stopping=True,
            max_iter=300,
            random_state=RANDOM_STATE,
        )),
    ]),
} 

## 7. Fonction d'evaluation

On utilise une validation croisee stratifiee pour verifier la stabilite, puis on evalue sur le test set.

In [ ]:
def evaluate_model(name, model):
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scoring = {
        'roc_auc': 'roc_auc',
        'recall': 'recall',
        'f1': 'f1',
        'precision': 'precision',
    }

    start = perf_counter()
    cv_scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        error_score='raise',
    )
    model.fit(X_train, y_train)
    duration = perf_counter() - start

    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= THRESHOLD).astype(int)

    return {
        'modele': name,
        'cv_roc_auc_mean': cv_scores['test_roc_auc'].mean(),
        'cv_recall_mean': cv_scores['test_recall'].mean(),
        'cv_f1_mean': cv_scores['test_f1'].mean(),
        'cv_precision_mean': cv_scores['test_precision'].mean(),
        'test_roc_auc': roc_auc_score(y_test, proba),
        'test_accuracy': accuracy_score(y_test, pred),
        'test_precision': precision_score(y_test, pred, zero_division=0),
        'test_recall': recall_score(y_test, pred),
        'test_f1': f1_score(y_test, pred),
        'temps_entrainement_secondes': duration,
    }

## 8. Entrainement et sauvegarde des modeles

In [ ]:
rows = []
trained_models = {}

for name, model in models.items():
    print('Entrainement :', name)
    rows.append(evaluate_model(name, model))
    trained_models[name] = model
    joblib.dump(model, MODELS_DIR / f'model_{name}.pkl')

comparison = pd.DataFrame(rows).sort_values(['test_recall', 'test_roc_auc', 'test_f1'], ascending=False)
comparison.to_csv(REPORTS_DIR / 'model_comparison.csv', index=False)

best_name = comparison.iloc[0]['modele']
joblib.dump(trained_models[best_name], MODELS_DIR / 'best_model.pkl')

comparison

## 9. Analyse du seuil

Le seuil principal reste 0.35. On garde ce tableau pour comprendre l'impact du seuil, mais le choix final n'est pas base sur une baisse artificielle du seuil.

In [ ]:
threshold_rows = []
for name, model in trained_models.items():
    proba = model.predict_proba(X_test)[:, 1]
    for threshold in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
        pred = (proba >= threshold).astype(int)
        threshold_rows.append({
            'modele': name,
            'threshold': threshold,
            'recall': recall_score(y_test, pred),
            'precision': precision_score(y_test, pred, zero_division=0),
            'f1': f1_score(y_test, pred),
            'accuracy': accuracy_score(y_test, pred),
            'clients_alertes': int(pred.sum()),
        })

thresholds = pd.DataFrame(threshold_rows).sort_values(['recall', 'precision', 'f1'], ascending=False)
thresholds.to_csv(REPORTS_DIR / 'threshold_analysis.csv', index=False)
thresholds.head(12)

## 10. Conclusion

Le modele retenu est celui qui obtient le meilleur recall au seuil fixe 0.35. Dans nos resultats, c'est la Logistic Regression.

C'est coherent avec l'objectif metier : detecter le maximum de clients qui risquent vraiment de partir.